In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [2]:
!kaggle datasets download -d elemento/nyc-yellow-taxi-trip-data


Dataset URL: https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data
License(s): U.S. Government Works
 99% 1.76G/1.78G [00:16<00:00, 159MB/s]
100% 1.78G/1.78G [00:16<00:00, 115MB/s]


In [4]:
import zipfile
import os

zip_path = "/content/nyc-yellow-taxi-trip-data.zip"
extract_path = "/content/nyc_taxi_data"

# فك الضغط
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(" تم فك الضغط بنجاح!")
print(" الملفات الموجودة:", os.listdir(extract_path))


 تم فك الضغط بنجاح!
 الملفات الموجودة: ['yellow_tripdata_2016-02.csv', 'yellow_tripdata_2016-03.csv', 'yellow_tripdata_2015-01.csv', 'yellow_tripdata_2016-01.csv']


In [5]:
import os
import pandas as pd
import dask.dataframe as dd
import gzip
import shutil
import time


In [6]:
data_path = "/content/nyc_taxi_data"
csv_files = [os.path.join(data_path, f) for f in os.listdir(data_path) if f.endswith(".csv")]
print(f"عدد الملفات: {len(csv_files)}")


عدد الملفات: 4


In [7]:
combined_file = "/content/combined_taxi_data.csv"

with open(combined_file, 'wb') as outfile:
    for i, file in enumerate(csv_files):
        with open(file, 'rb') as infile:
            if i != 0:
                infile.readline()  # تجنّب تكرار العنوان (Header)
            shutil.copyfileobj(infile, outfile)

print(f" تم الدمج! الحجم الإجمالي: {os.path.getsize(combined_file)/1e9:.2f} GB")


 تم الدمج! الحجم الإجمالي: 7.39 GB


In [10]:
import pandas as pd
import time
import os

combined_file = "/content/combined_taxi_data.csv"

start_time = time.time()

chunks = pd.read_csv(combined_file, chunksize=5000)
rows = 0
for chunk in chunks:
    rows += len(chunk)

end_time = time.time()

storage_gb = os.path.getsize(combined_file) / 1e9

print(f" Total rows (Pandas): {rows}")
print(f" Pandas (chunksize) time: {end_time - start_time:.2f} seconds")
print(f" Storage size: {storage_gb:.2f} GB")


 Total rows (Pandas): 47248845
 Pandas (chunksize) time: 202.32 seconds
 Storage size: 7.39 GB


In [12]:
import dask.dataframe as dd
import time
import os

combined_file = "/content/combined_taxi_data.csv"

start_time = time.time()

df_dask = dd.read_csv(combined_file, assume_missing=True)
rows_dask = len(df_dask)
df_dask.head()

end_time = time.time()

storage_gb = os.path.getsize(combined_file) / 1e9

print(f" Total rows (Dask): {rows_dask}")
print(f"Dask read time: {end_time - start_time:.2f} seconds")
print(f" Storage size: {storage_gb:.2f} GB")


 Total rows (Dask): 47248845
Dask read time: 192.62 seconds
 Storage size: 7.39 GB


In [13]:
import pandas as pd
import time
import gzip
import shutil
import os

combined_file = "/content/combined_taxi_data.csv"
compressed_file = "/content/combined_taxi_data.csv.gz"

# ضغط الملف
with open(combined_file, 'rb') as f_in:
    with gzip.open(compressed_file, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

compressed_size_gb = os.path.getsize(compressed_file) / 1e9
print(f"📦 Compressed size: {compressed_size_gb:.2f} GB")

start_time = time.time()

with gzip.open(compressed_file, 'rt') as f:
    df_gzip = pd.read_csv(f, nrows=1000)

end_time = time.time()

print(f" Gzip read success! Rows: {len(df_gzip)}")
print(f" Gzip read time: {end_time - start_time:.2f} seconds")


📦 Compressed size: 1.84 GB
 Gzip read success! Rows: 1000
 Gzip read time: 0.15 seconds
